# 01 - Exploration du dataset PaySim
Objectif : comprendre les données avant la modélisation (structure, déséquilibre des classes, fraudes par type, comportement dans le temps, soldes, comptes).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Charger le fichier CSV (placé dans data/)
df = pd.read_csv("../data/paysim.csv")

## 1. Aperçu général

In [ ]:
# Premières transactions
print(df.head())

# Nombre de lignes et de colonnes
print(df.shape)

# Types de variables
print(df.dtypes)

# Valeurs manquantes
print(df.isna().sum())

## 2. Déséquilibre des classes

In [ ]:
# Nombre de transactions normales et frauduleuses
print(df["isFraud"].value_counts())

# Pourcentage de chaque classe
print(df["isFraud"].value_counts(normalize=True) * 100)

ax = sns.countplot(x="isFraud", data=df)
ax.set_yscale("log")
ax.set_title("Répartition des classes (échelle log)")
plt.show()

## 3. Fraudes par type de transaction

In [ ]:
# Fraudes par type de transaction
print(pd.crosstab(df["type"], df["isFraud"]))

# Taux de fraude par type (%)
taux = df.groupby("type")["isFraud"].mean() * 100
print(taux)

taux.sort_values().plot(kind="barh", title="Taux de fraude par type (%)")
plt.show()

## 4. Montants

In [ ]:
print(df.groupby("isFraud")["amount"].describe())

plt.figure(figsize=(7, 4))
sns.boxplot(x="isFraud", y="amount", data=df, showfliers=False)
plt.title("Montant selon la classe (sans valeurs extrêmes)")
plt.show()

## 5. Dimension temporelle
`step` = 1 heure (743 steps, environ 30 jours).

In [ ]:
df["hour"] = df["step"] % 24
df["day"] = df["step"] // 24

fraud = df[df["isFraud"] == 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fraud.groupby("hour").size().plot(ax=axes[0], kind="bar", title="Fraudes par heure de la journée")
fraud.groupby("day").size().plot(ax=axes[1], title="Fraudes par jour")
plt.tight_layout()
plt.show()

# Comparaison avec le volume total par heure
total_h = df.groupby("hour").size()
(fraud.groupby("hour").size() / total_h * 100).plot(kind="bar", title="Taux de fraude par heure (%)")
plt.show()

## 6. Soldes et incohérences

In [ ]:
# Compte émetteur vidé complètement
vide = (df["oldbalanceOrg"] > 0) & (df["newbalanceOrig"] == 0)
print("Taux de fraude si compte émetteur vidé :", df.loc[vide, "isFraud"].mean() * 100)
print("Taux de fraude sinon :", df.loc[~vide, "isFraud"].mean() * 100)

# Écarts de cohérence des soldes
df["errorOrig"] = df["newbalanceOrig"] + df["amount"] - df["oldbalanceOrg"]
df["errorDest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
print(df.groupby("isFraud")[["errorOrig", "errorDest"]].describe().T)

## 7. Comptes

In [ ]:
print("Émetteurs uniques :", df["nameOrig"].nunique())
print("Destinataires uniques :", df["nameDest"].nunique())

# Comptes les plus actifs (émetteurs)
print(df["nameOrig"].value_counts().head(15))

# Comptes impliqués dans des fraudes
print(fraud["nameOrig"].nunique(), "émetteurs frauduleux")
print(fraud["nameDest"].nunique(), "destinataires frauduleux")

## 8. Premier graphe transactionnel (échantillon de fraudes)

In [ ]:
import networkx as nx

sample = fraud.sample(min(60, len(fraud)), random_state=42)
G = nx.from_pandas_edgelist(
    sample, source="nameOrig", target="nameDest",
    edge_attr=["amount", "type", "step"], create_using=nx.DiGraph
)

plt.figure(figsize=(10, 7))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, node_size=60, arrowsize=8, with_labels=False)
plt.title("Graphe d'un échantillon de transactions frauduleuses")
plt.show()

print("Nœuds :", G.number_of_nodes(), "| Arêtes :", G.number_of_edges())

## 9. Sous-ensemble pour la démo (10 à 12 comptes)
Les comptes de PaySim apparaissent peu souvent. On prend les comptes impliqués dans le plus de transactions (en émission ou réception).

In [ ]:
compte_activite = pd.concat([df["nameOrig"], df["nameDest"]]).value_counts()
comptes_demo = compte_activite.head(12).index.tolist()
print(comptes_demo)

demo = df[df["nameOrig"].isin(comptes_demo) | df["nameDest"].isin(comptes_demo)]
print(demo.shape)
print(demo["isFraud"].value_counts())

## Conclusions de l'exploration
- Classes très déséquilibrées : privilégier recall, F1 et AUC-PR (pas l'accuracy).
- Fraudes concentrées sur certains types : à vérifier avec le tableau croisé.
- Variables utiles pressenties : type, montant, soldes avant/après, heure, comptes vidés, structure du graphe.
- Prochaine étape : feature engineering puis modèles (régression logistique, Random Forest, XGBoost).